<a href="https://colab.research.google.com/github/tassid/relacoes_fisicas_ambiente/blob/main/codigos_python/trabalho_aula_01_sudoeste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho da Aula 01 · Área de estudo do grupo
### Pato Branco, Francisco Beltrão e Dois Vizinhos (Sudoeste do Paraná)

**Disciplina:** Relações Físicas do Ambiente Agrícola (PPGTCA/UTFPR)

## 1. Autenticação
___

Cada integrante usa o **próprio projeto** do Google Cloud registrado no Earth Engine. Troque `QUEM_ESTA_RODANDO` pelo seu nome.

In [ ]:
#!pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd

PROJETOS = {
    'Tassiane': 'ppgtca-relacoes-fisicas',
    'Eduardo':  'eduproject-510923',
    # 'Nome': 'id-do-projeto',   # demais integrantes
}
QUEM_ESTA_RODANDO = 'Tassiane'
ID_PROJETO = PROJETOS[QUEM_ESTA_RODANDO]

ee.Authenticate()
ee.Initialize(project=ID_PROJETO)
print('Earth Engine conectado ao projeto:', ID_PROJETO)

## 2. Limitando uma região
___

A área de estudo é formada por três municípios do **Sudoeste do Paraná**: **Pato Branco**, **Francisco Beltrão** e **Dois Vizinhos**. Eles estão no Terceiro Planalto Paranaense, sobre solos derivados de basalto, numa região de relevo ondulado e de transição entre os climas Cfa (verão quente) e Cfb (verão ameno), ambos subtropicais úmidos e sem estação seca definida.

**Justificativa agronômica**

- **Cultura predominante:** a agricultura é feita principalmente em propriedades familiares. O sistema típico é **soja ou milho no verão** e **trigo ou culturas de cobertura no inverno**. A região também é uma das principais bacias leiteiras do estado e tem forte produção de aves e suínos, que dependem do milho e das pastagens produzidos ali.
- **Importância regional:** Pato Branco, Francisco Beltrão e Dois Vizinhos são os polos do Sudoeste, com cooperativas, agroindústrias de carnes e leite e câmpus da UTFPR com cursos de agronomia e zootecnia. O clima desses municípios afeta tanto a produção de grãos quanto a de proteína animal da região.
- **Problema climático de interesse:** apesar de ser uma das regiões mais chuvosas do estado, o Sudoeste sofre com **estiagens** no verão, como na safra 2021/22, que afetam a soja, o milho e as pastagens. No inverno, as **geadas** são mais frequentes que no norte do estado e limitam o trigo e o milho semeado tarde. O **excesso de chuva** na colheita é outro risco.
- **Por que três municípios:** eles são próximos entre si, mas formam um gradiente de altitude, com Pato Branco no ponto mais alto. Isso permite comparar locais com clima parecido e testar se modelos calibrados em um município valem para os outros.

Na base FAO/GAUL os nomes estão **sem acento** (`Francisco Beltrao`).

In [ ]:
# Período da chuva (anos inclusivos)
ANO_INICIO_CHUVA = 1990
ANO_FIM_CHUVA = 2024
N_ANOS_CHUVA = ANO_FIM_CHUVA - ANO_INICIO_CHUVA + 1

municipios = ee.FeatureCollection('FAO/GAUL/2015/level2')


def definir_regiao(estado, municipio):
  area_estudo = (municipios
                 .filter(ee.Filter.eq('ADM1_NAME', estado))
                 .filter(ee.Filter.eq('ADM2_NAME', municipio)))
  geom = area_estudo.geometry()
  return [area_estudo.size().getInfo(), area_estudo, geom]


pato_branco = definir_regiao('Parana', 'Pato Branco')
francisco_beltrao = definir_regiao('Parana', 'Francisco Beltrao')
dois_vizinhos = definir_regiao('Parana', 'Dois Vizinhos')

print('Feições encontradas - Pato Branco:', pato_branco[0])
print('Feições encontradas - Francisco Beltrão:', francisco_beltrao[0])
print('Feições encontradas - Dois Vizinhos:', dois_vizinhos[0])

# As três cidades juntas, usadas no mapa conjunto e nos mapas de chuva e altitude
tres_cidades = pato_branco[1].merge(francisco_beltrao[1]).merge(dois_vizinhos[1])
geom_tres = tres_cidades.geometry()

## 3. Mapas interativos
___

### 3.1 Cada município

In [ ]:
def iniciar_mapa(data, propriedades, descricao):
  mapa = geemap.Map()
  mapa.add_basemap('HYBRID')
  mapa.addLayer(data[1], propriedades, descricao)
  mapa.centerObject(data[1], 11)
  return mapa

In [ ]:
pato_branco_mapa = iniciar_mapa(pato_branco, {'color': 'violet'}, 'Limite de Pato Branco, PR')
pato_branco_mapa

In [ ]:
francisco_beltrao_mapa = iniciar_mapa(francisco_beltrao, {'color': 'gold'}, 'Limite de Francisco Beltrão, PR')
francisco_beltrao_mapa

In [ ]:
dois_vizinhos_mapa = iniciar_mapa(dois_vizinhos, {'color': 'orange'}, 'Limite de Dois Vizinhos, PR')
dois_vizinhos_mapa

### 3.2 As três cidades no mesmo mapa

In [ ]:
CIDADES = [
  (pato_branco, 'Pato Branco', 'violet'),
  (francisco_beltrao, 'Francisco Beltrão', 'gold'),
  (dois_vizinhos, 'Dois Vizinhos', 'orange'),
]


def contornos(mapa):
  # Desenha só o contorno de cada município, para não esconder as camadas de baixo
  for data, nome, cor in CIDADES:
    mapa.addLayer(ee.Image().paint(data[1], 0, 3), {'palette': [cor]}, f'Limite de {nome}')


mapa_tres = geemap.Map()
mapa_tres.add_basemap('HYBRID')
contornos(mapa_tres)
HEX = {'violet': '#EE82EE', 'gold': '#FFD700', 'orange': '#FFA500'}   # mesmas cores, em hexadecimal, para a legenda
mapa_tres.add_legend(title='Municípios', legend_dict={n: HEX[c] for _, n, c in CIDADES})
mapa_tres.centerObject(tres_cidades, 9)
mapa_tres

### 3.3 Mapa de altitude (SRTM, 30 m)

In [ ]:
srtm = ee.Image('USGS/SRTMGL1_003')
vis_altitude = {'min': 350, 'max': 950, 'palette': ['006837', 'a6d96a', 'fee08b', 'f46d43', 'a50026']}

mapa_altitude = geemap.Map()
mapa_altitude.add_basemap('HYBRID')
mapa_altitude.addLayer(srtm.clip(geom_tres), vis_altitude, 'Altitude (m)')
contornos(mapa_altitude)
mapa_altitude.add_colorbar(vis_altitude, label='Altitude (m)')
mapa_altitude.centerObject(tres_cidades, 9)
mapa_altitude

### 3.4 Mapa de precipitação (CHIRPS)

Chuva **média anual** no período escolhido: soma das imagens diárias dividida pelo número de anos.

In [ ]:
chirps_periodo = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                  .filterDate(f'{ANO_INICIO_CHUVA}-01-01', f'{ANO_FIM_CHUVA + 1}-01-01'))
chuva_media_anual = chirps_periodo.sum().divide(N_ANOS_CHUVA)
vis_chuva = {'min': 1700, 'max': 2300, 'palette': ['f7fbff', 'c6dbef', '6baed6', '2171b5', '08306b']}

mapa_chuva = geemap.Map()
mapa_chuva.add_basemap('HYBRID')
mapa_chuva.addLayer(chuva_media_anual.clip(geom_tres), vis_chuva, 'Chuva média anual (mm/ano)')
contornos(mapa_chuva)
mapa_chuva.add_colorbar(vis_chuva, label=f'Chuva média anual {ANO_INICIO_CHUVA}–{ANO_FIM_CHUVA} (mm/ano)')
mapa_chuva.centerObject(tres_cidades, 9)
mapa_chuva

> 💡 Se as cores do mapa de chuva ficarem todas iguais, ajuste `min` e `max` em `vis_chuva` com base nos valores da tabela da seção 4. O pixel do CHIRPS tem ~5,5 km, por isso o mapa aparece em blocos.

## 4. Três informações
___

In [ ]:
# Área total
def area_total(data):
  area = data[2].area().divide(1e6).getInfo()
  return area

In [ ]:
pato_branco_area = area_total(pato_branco)
print(f'Área total - Pato Branco/PR: {pato_branco_area:.1f} km²')

francisco_beltrao_area = area_total(francisco_beltrao)
print(f'Área total - Francisco Beltrão/PR: {francisco_beltrao_area:.1f} km²')

dois_vizinhos_area = area_total(dois_vizinhos)
print(f'Área total - Dois Vizinhos/PR: {dois_vizinhos_area:.1f} km²')

In [ ]:
# Altitude média
def altitude(data):
  altitude_media = srtm.reduceRegion(reducer=ee.Reducer.mean(), geometry=data[2], scale=30, maxPixels=1e9).get('elevation').getInfo()
  return altitude_media

In [ ]:
pato_branco_altitude = altitude(pato_branco)
print(f'Altitude média - Pato Branco/PR: {pato_branco_altitude:.1f} m')

francisco_beltrao_altitude = altitude(francisco_beltrao)
print(f'Altitude média - Francisco Beltrão/PR: {francisco_beltrao_altitude:.1f} m')

dois_vizinhos_altitude = altitude(dois_vizinhos)
print(f'Altitude média - Dois Vizinhos/PR: {dois_vizinhos_altitude:.1f} m')

In [ ]:
# Chuva total no período (usa a geometria do próprio município)
def chuva_total(data, ANO_INICIO_CHUVA, ANO_FIM_CHUVA, LOCALIZACAO):
  chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
            .filterDate(f'{ANO_INICIO_CHUVA}-01-01', f'{ANO_FIM_CHUVA + 1}-01-01')
            .filterBounds(data[2]))
  chuva_total_mm = chirps.sum().reduceRegion(reducer=ee.Reducer.mean(), geometry=data[2], scale=5566, maxPixels=1e9).get('precipitation').getInfo()
  n_anos = ANO_FIM_CHUVA - ANO_INICIO_CHUVA + 1
  print(f'{LOCALIZACAO}: chuva total de {ANO_INICIO_CHUVA} a {ANO_FIM_CHUVA}: {chuva_total_mm:.0f} mm '
        f'(média de {chuva_total_mm / n_anos:.0f} mm/ano)')
  return chuva_total_mm

In [ ]:
chuva_total_pato_branco = chuva_total(pato_branco, ANO_INICIO_CHUVA, ANO_FIM_CHUVA, 'Pato Branco/PR')
chuva_total_francisco_beltrao = chuva_total(francisco_beltrao, ANO_INICIO_CHUVA, ANO_FIM_CHUVA, 'Francisco Beltrão/PR')
chuva_total_dois_vizinhos = chuva_total(dois_vizinhos, ANO_INICIO_CHUVA, ANO_FIM_CHUVA, 'Dois Vizinhos/PR')

In [ ]:
# Resumo das três informações
tabela = pd.DataFrame({
  'Área total (km²)': [pato_branco_area, francisco_beltrao_area, dois_vizinhos_area],
  'Altitude média (m)': [pato_branco_altitude, francisco_beltrao_altitude, dois_vizinhos_altitude],
  f'Chuva {ANO_INICIO_CHUVA}–{ANO_FIM_CHUVA} (mm)': [chuva_total_pato_branco, chuva_total_francisco_beltrao, chuva_total_dois_vizinhos],
}, index=['Pato Branco', 'Francisco Beltrão', 'Dois Vizinhos'])
tabela['Média anual (mm/ano)'] = tabela[f'Chuva {ANO_INICIO_CHUVA}–{ANO_FIM_CHUVA} (mm)'] / N_ANOS_CHUVA
tabela.round(1)

> 💬 **Para discutir:** a altitude média do município é menor que a da sede urbana, porque os vales dos rios entram na média. O CHIRPS combina satélite e estações; comparem a média anual com a da estação do INMET de cada cidade.

## 5. Pergunta agroclimática do grupo
___

> **No Sudoeste do Paraná, é possível estimar com precisão suficiente a demanda hídrica das culturas usando apenas a temperatura do ar e, com isso, explicar quanto da variação da produtividade da soja e do milho se deve ao déficit hídrico?**

Como a pergunta se encaixa nos projetos da disciplina:

| Projeto | Parte da pergunta |
|---|---|
| P1 · Elementos meteorológicos | Estimar a radiação solar e a ETo só com temperatura, e medir o erro dessa estimativa |
| P2 · Chuva, ETo e balanço hídrico | Usar essa ETo no balanço hídrico e identificar os períodos de deficiência nas safras de verão e de inverno |
| P3 · Produtividade | Penalizar a produtividade potencial pelo déficit hídrico (MZA-FAO) e comparar com a produtividade do IBGE |

### Perguntas complementares: El Niño-Oscilação Sul (ENOS)

O ENOS é a principal fonte de variabilidade climática de um ano para o outro no Sul do Brasil. Em geral, o **El Niño** está associado a mais chuva, principalmente na primavera, e a **La Niña** a estiagens, como na safra 2021/22. O Sudoeste do Paraná fica dentro dessa região de influência, então espera-se um sinal mais forte que no norte do estado.

1. **Chuva:** a chuva da primavera e do verão (outubro a março) nos três municípios é maior em anos de El Niño e menor em anos de La Niña, em comparação com anos neutros? A diferença é a mesma em Pato Branco, Francisco Beltrão e Dois Vizinhos?
2. **Radiação e ETo:** em anos de El Niño, a maior nebulosidade reduz a radiação solar global e a ETo? Os modelos de radiação por temperatura do P1 continuam funcionando bem nesses anos, ou o erro aumenta?
3. **Produtividade:** a produtividade da soja e do milho (IBGE) é menor nas safras de La Niña? Quanto dessa perda o déficit hídrico calculado pelo balanço hídrico (P2) consegue explicar?

Dados para responder: classificação dos anos pelo índice ONI da NOAA (El Niño, La Niña ou neutro), chuva do CHIRPS (desde 1981), radiação e ETo do BR-DWGD e produtividade da PAM/IBGE (SIDRA).